In [ ]:
# NOTEBOOK 02: SQL ANALYSIS

import pandas as pd
import sqlite3

# Cell 1: Load Data and Create Database
df = pd.read_excel('returns_dataset (1).xlsx', sheet_name='Returns_Data')
df['Support_Channel'] = df['Support_Channel'].fillna('No Support Contacted')

conn = sqlite3.connect(':memory:')
df.to_sql('returns', conn, index=False)
print("✅ Database ready! Table 'returns' loaded with 5000 rows")

# Cell 2: Query 1 - Avoidable Return Rate by Category
q1 = """
SELECT
    Product_Category,
    COUNT(*) AS Total_Returns,
    SUM(Is_Avoidable_Return) AS Avoidable_Returns,
    ROUND(SUM(Is_Avoidable_Return) * 100.0 / COUNT(*), 1) AS Avoidable_Pct
FROM returns
GROUP BY Product_Category
ORDER BY Avoidable_Pct DESC;
"""
print("\n--- Avoidable Return Rate by Category ---")
print(pd.read_sql(q1, conn).to_string(index=False))

# Cell 3: Query 2 - Support Channel Effectiveness
q2 = """
SELECT
    Support_Channel,
    COUNT(*) AS Total_Cases,
    SUM(Return_Prevented) AS Returns_Prevented,
    ROUND(SUM(Return_Prevented) * 100.0 / COUNT(*), 1) AS Prevention_Rate,
    ROUND(AVG(CSAT_Score), 2) AS Avg_CSAT
FROM returns
WHERE Support_Channel != 'No Support Contacted'
GROUP BY Support_Channel
ORDER BY Prevention_Rate DESC;
"""
print("\n--- Support Channel Effectiveness ---")
print(pd.read_sql(q2, conn).to_string(index=False))

# Cell 4: Query 3 - Revenue at Risk by Region
q3 = """
SELECT
    Region,
    COUNT(*) AS Avoidable_Returns,
    ROUND(SUM(Product_Price_INR), 2) AS Revenue_At_Risk,
    ROUND(AVG(Product_Price_INR), 2) AS Avg_Price
FROM returns
WHERE Is_Avoidable_Return = 1
GROUP BY Region
ORDER BY Revenue_At_Risk DESC;
"""
print("\n--- Revenue at Risk by Region ---")
print(pd.read_sql(q3, conn).to_string(index=False))

# Cell 5: Query 4 - Monthly Return Trends
q4 = """
SELECT
    SUBSTR(Return_Request_Date, 1, 7) AS Month,
    COUNT(*) AS Total_Returns,
    SUM(Is_Avoidable_Return) AS Avoidable,
    SUM(Return_Prevented) AS Prevented
FROM returns
GROUP BY Month
ORDER BY Month;
"""
print("\n--- Monthly Return Trends ---")
print(pd.read_sql(q4, conn).to_string(index=False))

# Cell 6: Query 5 - Top Brands with Avoidable Returns
q5 = """
SELECT
    Brand,
    Product_Category,
    COUNT(*) AS Total_Returns,
    SUM(Is_Avoidable_Return) AS Avoidable,
    ROUND(SUM(Is_Avoidable_Return) * 100.0 / COUNT(*), 1) AS Avoidable_Pct
FROM returns
GROUP BY Brand, Product_Category
ORDER BY Avoidable DESC
LIMIT 10;
"""
print("\n--- Top 10 Brand + Category with Most Avoidable Returns ---")
print(pd.read_sql(q5, conn).to_string(index=False))

conn.close()

print("\n" + "=" * 55)
print("   NOTEBOOK 02 COMPLETE")
print("=" * 55)

✅ Database ready! Table 'returns' loaded with 5000 rows

--- Avoidable Return Rate by Category ---
Product_Category  Total_Returns  Avoidable_Returns  Avoidable_Pct
         Cameras            329                157           47.7
        Speakers            249                118           47.4
        Monitors            372                176           47.3
    Smartwatches            395                186           47.1
         Laptops            883                415           47.0
 Gaming Consoles            345                157           45.5
      Headphones            512                229           44.7
         Tablets            614                274           44.6
     Smartphones           1020                444           43.5
        Printers            281                116           41.3

--- Support Channel Effectiveness ---
   Support_Channel  Total_Cases  Returns_Prevented  Prevention_Rate  Avg_CSAT
         Live Chat          687                139        